# Task 2.2: Reproduction of Latent Hierarchical Structural Learning
**Contribution reproduced:** The joint latent structural inference (Eq 5) and iCCCP learning algorithm (Fig 4).
**Evaluation metric:** Accuracy.

First, define the features and structural components.

In [1]:
import numpy as np
from sklearn.svm import LinearSVC
np.random.seed(42)
X_train, y_train = np.load('data/X_train.npy'), np.load('data/y_train.npy')
X_test, y_test = np.load('data/X_test.npy'), np.load('data/y_test.npy')


The code block below extracts appearance $\Phi_A$ and shape $\Phi_S$ features. It references **Equation 4** from the paper.

In [2]:
def extract_phi_A(image, p_y, p_x, size):
    return image[p_y:p_y+size, p_x:p_x+size].flatten()

def extract_phi_S(p_y, p_x, ref_y, ref_x):
    dy, dx = p_y - ref_y, p_x - ref_x
    return np.array([dy, dx, dy**2, dx**2])

def get_feature_vector(image, part_positions):
    # 3-layer model simplification: We model 5 parts (Layer 2) for this toy demo
    # part_positions: list of (y,x) tuples for Center, Top, Bot, Left, Right
    phi_A = [extract_phi_A(image, y, x, 4) for (y,x) in part_positions]
    ref_positions = [(4,4), (0,4), (8,4), (4,0), (4,8)]
    phi_S = [extract_phi_S(py, px, ry, rx) for (py, px), (ry, rx) in zip(part_positions, ref_positions)]
    return np.concatenate([np.concatenate(phi_A), np.concatenate(phi_S)])


The code block below performs Inference / Dynamic Programming to find the latent variables $h^*$. This corresponds to **Equation 5** and Step (1) of the CCCP algorithm in **Figure 3**.

In [3]:
def find_latent_h(image, w):
    if w is None:
        # Trivial initialization (p=0 displacement) as described in Section 4.3
        return [(4,4), (0,4), (8,4), (4,0), (4,8)]
    
    best_score = -np.inf
    best_pos = None
    # Very simplified discrete search for "Dynamic Programming"
    # Only jitter the 4 outer blocks by -1 to 1
    for dy_t in [-1,0,1]:
       for dy_b in [-1,0,1]:
           for dx_l in [-1,0,1]:
               for dx_r in [-1,0,1]:
                   pos = [(4,4), (0+dy_t,4), (8+dy_b,4), (4,0+dx_l), (4,8+dx_r)]
                   # valid check
                   if any(y<0 or y>8 or x<0 or x>8 for y,x in pos): continue
                   feat = get_feature_vector(image, pos)
                   score = np.dot(w, feat)
                   if score > best_score:
                       best_score = score
                       best_pos = pos
    return best_pos


The block below implements the Incremental Concave-Convex Procedure (iCCCP). This corresponds to **Figure 4**. It iteratively adds negative samples and solves the standard SVM given imputed latent variables.

In [4]:
def iCCCP_train(X_pos, X_neg):
    w = None
    model = LinearSVC(C=0.005, dual=False)
    n_neg = 30 # Initial negative set size as per Section 3.4
    K = 1.15
    
    for t in range(5): # Run 5 iCCCP iterations
        # Step 1: Fill in latent variables for current positives
        pos_feats = [get_feature_vector(x, find_latent_h(x, w)) for x in X_pos]
        
        # Fix latent variables for negatives (trivial locations for background)
        current_n_neg = min(len(X_neg), int(n_neg * (K**t)))
        neg_feats = [get_feature_vector(x, [(4,4), (0,4), (8,4), (4,0), (4,8)]) for x in X_neg[:current_n_neg]]
        
        X_iter = np.vstack([pos_feats, neg_feats])
        y_iter = np.array([1]*len(pos_feats) + [-1]*len(neg_feats))
        
        # Step 2: Solve structural SVM (here reduced to standard linear SVM on imputed features)
        model.fit(X_iter, y_iter)
        w = model.coef_[0]
        print(f"Iteration {t+1}: Trained with {current_n_neg} hard negatives.")
        
    return model, w

X_pos_train, X_neg_train = X_train[y_train==1], X_train[y_train==-1]
model_full, w_full = iCCCP_train(X_pos_train, X_neg_train)


Iteration 1: Trained with 30 hard negatives.
Iteration 2: Trained with 34 hard negatives.
Iteration 3: Trained with 39 hard negatives.
Iteration 4: Trained with 45 hard negatives.
Iteration 5: Trained with 52 hard negatives.
